In [120]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
# matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus']=False


import pandas as pd
from sklearn import svm,metrics
from scipy.special import expit
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor 
from sklearn.linear_model import LinearRegression,Ridge,Lasso,LogisticRegression
from sklearn.preprocessing import PolynomialFeatures,StandardScaler,OneHotEncoder
from sklearn.metrics import mean_absolute_error # 평균절대값오차 오차범위
from sklearn.model_selection import train_test_split
import numpy as np
from io import StringIO

In [55]:
# 2개만 분류 - 빙어/도미
# KNeighborsClassifier - 다중분류
# 데이터 불러오기
fish =pd.read_csv('data/fish_logistic.csv')
fish.head(2)
fish['Species'].unique() # 7개의 종류를 가지고 있음

<StringArray>
['Bream', 'Roach', 'Whitefish', 'Parkki', 'Perch', 'Pike', 'Smelt']
Length: 7, dtype: str

In [56]:
# fish.info()
# fish.describe()

In [57]:
fish.columns

Index(['Species', 'Weight', 'Length', 'Diagonal', 'Height', 'Width'], dtype='str')

In [58]:
# 데이터 전처리
data =fish[fish.columns[1:]].values # 1번째꺼 빼고 다
label = fish['Species']


In [59]:
# 학습데이터, 테스트데이터 분리
train_data,test_data,train_label,test_label = train_test_split(data,label,random_state=42)


In [60]:
# LogisticRegression,KNN 은 무조건 정규화 작업이 필요함
ss = StandardScaler()
ss.fit(train_data)
train_scaled = ss.transform(train_data)
test_scaled = ss.transform(test_data)

In [61]:
test_label

78         Perch
155        Smelt
128         Pike
55     Whitefish
94         Perch
29         Bream
147        Smelt
51         Roach
98         Perch
141         Pike
19         Bream
60     Whitefish
15         Bream
65        Parkki
24         Bream
30         Bream
126        Perch
101        Perch
96         Perch
16         Bream
151        Smelt
18         Bream
12         Bream
9          Bream
31         Bream
125        Perch
95         Perch
56     Whitefish
145        Smelt
152        Smelt
135         Pike
76         Perch
75         Perch
138         Pike
2          Bream
86         Perch
45         Roach
42         Roach
68        Parkki
118        Perch
Name: Species, dtype: str

In [62]:
### 이진분류 형태 구성
### Bream,Smelt 생선만 분리 # 테스트용
# 갯수비교를 항상 해야함
bream_smelt_index = (train_label=='Bream') | (train_label =='Smelt')
# ※ True/False 목록 → True인 행(Bream, Smelt)만 골라냄
train_bream_smelt = train_scaled[bream_smelt_index]
label_bream_smelt = train_label[bream_smelt_index]
label_bream_smelt.shape
bream_smelt_index.value_counts()


bream_smelt_index1 = (test_label=='Bream') | (test_label =='Smelt')
test_bream_smelt = test_scaled[bream_smelt_index1]
label_bream_smelt1 = test_label[bream_smelt_index1]

In [63]:
# 데이터 학습
lr = LogisticRegression()
lr.fit(train_bream_smelt,label_bream_smelt)
print(lr.classes_)
# 데이터 예측
predict = lr.predict(train_bream_smelt[:5])
print(lr.predict_proba(train_bream_smelt[:5]))
# 정확도 - train
print('train',lr.score(train_bream_smelt,label_bream_smelt))
print('test',lr.score(test_bream_smelt,label_bream_smelt1))

['Bream' 'Smelt']
[[0.99760007 0.00239993]
 [0.02737325 0.97262675]
 [0.99486386 0.00513614]
 [0.98585047 0.01414953]
 [0.99767419 0.00232581]]
train 1.0
test 1.0


In [68]:
train_bream_smelt[:1]
fish.columns
in_data = train_data[bream_smelt_index]
in_data[:5]

array([[720.    ,  35.    ,  40.6   ,  16.3618,   6.09  ],
       [  7.5   ,  10.5   ,  11.6   ,   1.972 ,   1.16  ],
       [620.    ,  34.5   ,  39.7   ,  15.5227,   5.2801],
       [500.    ,  31.    ,  36.2   ,  14.3714,   4.8146],
       [714.    ,  36.    ,  41.5   ,  16.517 ,   5.8515]])

In [114]:
# 데이터를 입력받아 생선을 분류하시오
# 분류생선확률 : Bream, (99.76%)
# 컬럼 'Weight', 'Length', 'Diagonal', 'Height', 'Width'
# 720,35,40.6,16.3618,6.09
# 7.5,10.5,11.6,1.972,1.16
in_data = input('데이터 입력:')
in_data1 = list(map(float,in_data.split(',')))

ss.fit([in_data1])
# ※ 삭제할 것! 입력값으로 fit하면 정규화가 0이 돼서 항상 같은 결과
t_scaled = ss.transform([in_data1])



result = lr.predict(t_scaled)
result2 = lr.predict_proba(t_scaled)

if result[0] =='Bream':
    result3 = result2[0][0]
else:
    result3 = result2[0][1]

result3
print(f'분류생선확률 : {result[0]},({result3*100:.2f}%)')

분류생선확률 : Bream,(89.68%)


In [115]:
print(lr.coef_,lr.intercept_) # LogisticRegression 선그래프로 표현 가능


[[-0.40451732 -0.57582787 -0.66248158 -1.01329614 -0.73123131]] [-2.16172774]


In [117]:
# 결과값 z 의 값을 출력하는 함수
decsions = lr.decision_function(train_bream_smelt[:5])
decsions

array([-6.02991358,  3.57043428, -5.26630496, -4.24382314, -6.06135688])

In [ ]:
# 2진분류에 사용되는  로직스틱함수(0-1까지의 값으로 변경해서 보여줌)

print(expit(decsions))
# ※ z값 → 0~1 확률. predict_proba의 두 번째 칸(Smelt)과 같음

[0.00239993 0.97262675 0.00513614 0.01414953 0.00232581]
